In [11]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [12]:
df = pd.read_csv('loan.csv')

df

,age,gender,occupation,education_level,marital_status,income,credit_score,loan_status
0,32,Male,Engineer,Bachelor's,Married,85000,720,Approved
1,45,Female,Teacher,Master's,Single,62000,680,Approved
2,28,Male,Student,High School,Single,25000,590,Denied
3,51,Female,Manager,Bachelor's,Married,105000,780,Approved
4,36,Male,Accountant,Bachelor's,Married,75000,710,Approved
...,...,...,...,...,...,...,...,...
56,39,Male,Architect,Master's,Married,100000,770,Approved
57,25,Female,Receptionist,High School,Single,32000,570,Denied
58,43,Male,Banker,Bachelor's,Married,95000,760,Approved
59,30,Female,Writer,Master's,Single,55000,650,Approved


1.Get basic averages 

In [13]:
print("Average Income:", df['income'].mean())
print("Average Credit Score:", df['credit_score'].mean())

Average Income: 78983.60655737705
Average Credit Score: 709.8360655737705


Task 2: Filter for High Income ApplicantsWrite a simple condition to see only applicants earning over $\$80,000$:

In [14]:
#Filter for high income over 80k
high_earners = df[df['income'] > 80000]

#Display the top 5 high earners 
print(high_earners.head())

   age  gender occupation education_level marital_status  income  \
0   32    Male   Engineer      Bachelor's        Married   85000   
3   51  Female    Manager      Bachelor's        Married  105000   
6   42    Male     Lawyer        Doctoral        Married  120000   
8   37    Male         IT        Master's        Married   92000   
9   48  Female     Doctor        Doctoral        Married  180000   

   credit_score loan_status  
0           720    Approved  
3           780    Approved  
6           790    Approved  
8           750    Approved  
9           820    Approved  


Task 3 : Count Approved VS Denied loans 

Use .value_counts() to see how many people were approved vs. denied:

In [15]:
#Count totals for loan status 
print(df['loan_status'].value_counts())

loan_status
Approved    45
Denied      16
Name: count, dtype: int64


Task 4: Compare Income by Loan Status

Group the data by loan_status to compare the average income of approved applicants versus denied applicants:

In [16]:
# Compare average income 
print(df.groupby('loan_status')['income'].mean())

loan_status
Approved    92955.555556
Denied      39687.500000
Name: income, dtype: float64


Task 5: Calculate Approval Rate as a Percentage
Instead of just counting approved vs. denied, calculate the exact percentage of approved loans.

In [17]:
#Calculate total rows 
total_applicants = len(df)

#Count approved loans 
approved_count = (df['loan_status'] == 'Approved').sum()

#Calculate percentage 
approval_rate = (approved_count / total_applicants) * 100
print("approval_rate:", round(approval_rate, 2), "%")

approval_rate: 73.77 %


Task 6: Filter with Multiple Conditions
Find applicants who are both Single AND have a credit score above 700.

In [19]:
# Use & for AND, | for OR. Always wrap conditions in parentheses!
single_high_credit = df[(df['marital_status'] == 'Single') & (df['credit_score'] > 700)]

print(single_high_credit[['age', 'income', 'credit_score', 'loan_status']])

    age  income  credit_score loan_status
25   28   75000           710    Approved


Task 7: Group By and Aggregate Multiple Metrics
Group the data by education_level and see the average income and average credit score for each group in one table.

In [20]:
#Group by education level and calculate mean for mutiple columns
education_summary = df.groupby('education_level')[['income', 'credit_score']].mean()
print(education_summary)

                        income  credit_score
education_level                             
Associate's       50833.333333    651.666667
Bachelor's        72347.826087    703.043478
Doctoral         127300.000000    796.000000
High School       37428.571429    601.428571
Master's          87600.000000    736.666667


Task 8: Find the Minimum and Maximum for Approved Applicants
Check the financial boundaries for people who got approved.

In [21]:
#Filter for approved loans 
approved_df = df[df['loan_status'] == 'Approved']

#Get min and max credit scores 
print("Lowest Approved Credit Score: ", approved_df['credit_score'].min())
print("Highest Approved credit Score: ", approved_df['credit_score'].max())
print("Lowest Approved income: ", approved_df['income'].min())

Lowest Approved Credit Score:  650
Highest Approved credit Score:  830
Lowest Approved income:  55000


Task 9: Create a Debt-to-Income / Credit-to-Income Ratio Column
Create a new column called credit_to_income that measures how much credit score an applicant has relative to their income.

In [22]:
#Create a new column by dividing two existing columns 
df['credit_to_income'] = df['credit_score'] / df['income']

#display the first 5 rows with the new column 
print(df[['income', 'credit_score', 'credit_to_income', 'loan_status']].head())

   income  credit_score  credit_to_income loan_status
0   85000           720          0.008471    Approved
1   62000           680          0.010968    Approved
2   25000           590          0.023600      Denied
3  105000           780          0.007429    Approved
4   75000           710          0.009467    Approved


Task 10: Categorise Applicants into Income Brackets using pd.cut()
Instead of looking at exact income numbers, group applicants into 3 simple buckets: Low, Medium, and High.

In [23]:
#Define bin boundaries and labels 
bins = [0, 50000, 90000 , 200000]
labels = ['Low', 'Medium', 'High']

#Create the income_bracket column 
df['income_bracket'] = pd.cut(df['income'], bins=bins, labels=labels)

#check how many applicants fall into each bracket 
print(df['income_bracket'].value_counts())

income_bracket
Medium    25
High      21
Low       15
Name: count, dtype: int64


Task 11: Create a Risk Flag Column with Conditional Logic (np.select)
Combine multiple columns to create a custom risk_tier flag (Low Risk, Medium Risk, High Risk).

In [26]:
import numpy as np 

# define conditions 
conditions = [
    (df['credit_score'] >= 720) & (df['income'] >= 75000), #low risk
    (df['credit_score'] >= 650) & (df['income'] >= 53500) #medium risk
]

#define corresponding outputs 
choices = ['Low Risk', 'Medium Risk']

#create the new column (defaulting to 'high risk')
df['risk_tier'] = np.select(conditions, choices, default="High Risk")

# Compare Risk Tier against actual Loan status 
print(pd.crosstab(df['risk_tier'], df['loan_status']))

loan_status  Approved  Denied
risk_tier                    
High Risk           0      16
Low Risk           32       0
Medium Risk        13       0


Task 12: Convert Binary Text to 1s and 0s (Encoding)
Machine learning models cannot read string words like 'Approved' or 'Denied'. Convert loan_status into a numeric column called approved_flag.

In [27]:
# Map Approved -> 1 and Denied -> 0
df['approved_flag'] = df['loan_status'].map({'Approved': 1, 'Denied': 0})

# Calculate the average approval rate directly using mean()
print("Approval Rate (Numeric Mean):", df['approved_flag'].mean())

Approval Rate (Numeric Mean): 0.7377049180327869
